# ComicVideoAI trên Google Colab

Notebook này clone gói `colab_package` từ GitHub, cài dependency, cấu hình `.env.local`, chạy Vite và mở public URL bằng Cloudflare Tunnel.

Repo mặc định: `https://github.com/tqtuan8788-ai/comic-video`


In [ ]:
#@title 1) Cấu hình repo và thư mục chạy
GITHUB_REPO = "https://github.com/tqtuan8788-ai/comic-video.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
REPO_DIR = "/content/comic-video"
PROJECT_SUBDIR = "colab_package"
PROJECT_DIR = f"{REPO_DIR}/{PROJECT_SUBDIR}"
APP_PORT = 3000
OMNIVOICE_PORT = 7861
print("Project dir:", PROJECT_DIR)


In [ ]:
#@title 2) Cài system packages: ffmpeg
import os, subprocess

def run(cmd, cwd=None, check=True):
    print(f"$ {cmd}")
    return subprocess.run(cmd, cwd=cwd, shell=True, check=check)

run("apt-get update -qq && apt-get install -y -qq ffmpeg")
run("ffmpeg -version | head -1")


In [ ]:
#@title 3) Clone hoặc cập nhật code
import os, subprocess, pathlib, textwrap, shutil

def run(cmd, cwd=None, check=True):
    print(f"$ {cmd}")
    return subprocess.run(cmd, cwd=cwd, shell=True, check=check)

if not os.path.exists(REPO_DIR):
    run(f"git clone --branch {BRANCH} --depth 1 {GITHUB_REPO} {REPO_DIR}")
else:
    run("git fetch --depth 1 origin {branch}".format(branch=BRANCH), cwd=REPO_DIR)
    run("git checkout {branch}".format(branch=BRANCH), cwd=REPO_DIR)
    run("git pull --ff-only origin {branch}".format(branch=BRANCH), cwd=REPO_DIR)

if not os.path.isdir(PROJECT_DIR):
    raise FileNotFoundError(f"Không thấy {PROJECT_DIR}. Hãy kiểm tra PROJECT_SUBDIR hoặc branch.")
os.chdir(PROJECT_DIR)
print("Ready:", os.getcwd())


In [ ]:
#@title 4) Cài Node.js dependency
import os, subprocess, shutil
os.chdir(PROJECT_DIR)

# Colab thường đã có Node.js. In version để kiểm tra.
run("node --version && npm --version")
run("npm ci --no-audit --no-fund")


In [ ]:
#@title 5) Tạo .env.local (điền key nếu có)
DEEPSEEK_API_KEY = ""  #@param {type:"string"}
GEMINI_API_KEY = ""  #@param {type:"string"}
OPENAI_API_KEY = ""  #@param {type:"string"}
GROQ_API_KEY = ""  #@param {type:"string"}
OPENROUTER_API_KEY = ""  #@param {type:"string"}
TTS_ELEVENLABS_KEY = ""  #@param {type:"string"}
USE_FREE_ONLY = False  #@param {type:"boolean"}

from pathlib import Path

env = {
    "PROVIDER_PRIORITY": "openai_codex,openai_codex_image,deepseek,pollinations,tts_free,tts_omnivoice,openai,groq,openrouter,sdxl_local,gemini,tts_gemini,tts_elevenlabs",
    "DEEPSEEK_API_KEY": DEEPSEEK_API_KEY,
    "DEEPSEEK_BASE_URL": "https://api.deepseek.com",
    "DEEPSEEK_MODEL_TEXT": "deepseek-chat",
    "GEMINI_ENABLED": "false",
    "GEMINI_API_KEY": GEMINI_API_KEY,
    "GOOGLE_API_KEY": GEMINI_API_KEY,
    "OPENAI_API_KEY": OPENAI_API_KEY,
    "GROQ_API_KEY": GROQ_API_KEY,
    "OPENROUTER_API_KEY": OPENROUTER_API_KEY,
    "POLLINATIONS_IMAGE_URL": "https://image.pollinations.ai/prompt",
    "POLLINATIONS_MODEL_IMAGE": "flux",
    "TTS_FREE_PROVIDER": "omnivoice",
    "TTS_FREE_URL": "/api/omnivoice/tts",
    "OMNIVOICE_PROXY_TARGET": f"http://127.0.0.1:{OMNIVOICE_PORT}",
    "TTS_ELEVENLABS_KEY": TTS_ELEVENLABS_KEY,
    "USE_FREE_ONLY": str(USE_FREE_ONLY).lower(),
}
Path(".env.local").write_text("\n".join(f"{k}={v}" for k, v in env.items()) + "\n", encoding="utf-8")
print("Đã ghi .env.local (không commit file này).")


In [ ]:
#@title 6) Tuỳ chọn: cài và chạy OmniVoice backend (có thể mất lâu)
START_OMNIVOICE = False  #@param {type:"boolean"}
OMNIVOICE_DEVICE = "cuda"  #@param ["cuda", "cpu"]

import os, subprocess, time, pathlib
os.chdir(PROJECT_DIR)

if START_OMNIVOICE:
    # omnivoice có thể tải model lớn từ Hugging Face. Bật GPU Runtime trong Colab để nhanh hơn.
    run("pip -q install -r requirements.txt")
    env = os.environ.copy()
    env.update({
        "OMNIVOICE_HOST": "127.0.0.1",
        "OMNIVOICE_PORT": str(OMNIVOICE_PORT),
        "OMNIVOICE_DEVICE": OMNIVOICE_DEVICE,
        "HF_HOME": "/content/hf-cache",
        "HUGGINGFACE_HUB_CACHE": "/content/hf-cache/hub",
    })
    log = open("omnivoice.log", "w")
    p = subprocess.Popen(["python3", "scripts/omnivoice-api-server.py"], env=env, stdout=log, stderr=subprocess.STDOUT)
    pathlib.Path("omnivoice.pid").write_text(str(p.pid))
    print("OmniVoice PID:", p.pid)
    time.sleep(3)
    run(f"curl -sS http://127.0.0.1:{OMNIVOICE_PORT}/health || true", check=False)
else:
    print("Bỏ qua OmniVoice. App vẫn chạy; TTS local sẽ báo chưa sẵn sàng nếu được gọi.")


In [ ]:
#@title 7a) Smoke test kết nối tạo ảnh miễn phí Pollinations
# Cell này kiểm tra nhanh Colab có ra internet tới dịch vụ tạo ảnh miễn phí hay không.
# Nếu fail ở đây thì image provider Pollinations sẽ không chạy ổn trong phiên Colab đó.
import os, pathlib
os.chdir(PROJECT_DIR)
run('python3 - <<"PY"
import urllib.request
url="https://image.pollinations.ai/prompt/simple%20vertical%20comic%20panel?width=256&height=384&model=flux&seed=123&nologo=true&enhance=false"
with urllib.request.urlopen(url, timeout=60) as r:
    data=r.read(64)
    print("pollinations_status", r.status, r.headers.get("content-type"), len(data))
PY')


In [ ]:
#@title 8) Build kiểm tra nhanh
os.chdir(PROJECT_DIR)
run("npm run build")


In [ ]:
#@title 9) Chạy Vite dev server
import os, subprocess, pathlib, time
os.chdir(PROJECT_DIR)

# Dừng server cũ nếu notebook được chạy lại.
for pid_file in ["comicvideoai-dev.pid"]:
    p = pathlib.Path(pid_file)
    if p.exists():
        old_pid = p.read_text().strip()
        if old_pid:
            subprocess.run(f"kill {old_pid} 2>/dev/null || true", shell=True)

log = open("comicvideoai-dev.log", "w")
proc = subprocess.Popen(["npm", "run", "dev", "--", "--host", "0.0.0.0", "--port", str(APP_PORT)], stdout=log, stderr=subprocess.STDOUT)
pathlib.Path("comicvideoai-dev.pid").write_text(str(proc.pid))
time.sleep(4)
print("Vite PID:", proc.pid)
run(f"curl -I http://127.0.0.1:{APP_PORT} || (tail -80 comicvideoai-dev.log; exit 1)")


In [ ]:
#@title 10) Mở public URL bằng Cloudflare Tunnel
import os, subprocess, time, re, pathlib, shutil
os.chdir(PROJECT_DIR)

if not shutil.which("cloudflared"):
    run("wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared")

# Dừng tunnel cũ nếu có.
pid_path = pathlib.Path("cloudflared.pid")
if pid_path.exists():
    subprocess.run(f"kill {pid_path.read_text().strip()} 2>/dev/null || true", shell=True)

tunnel_log = open("cloudflared.log", "w")
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{APP_PORT}", "--no-autoupdate"], stdout=tunnel_log, stderr=subprocess.STDOUT)
pid_path.write_text(str(tunnel.pid))

public_url = None
for _ in range(40):
    time.sleep(1)
    text = pathlib.Path("cloudflared.log").read_text(errors="ignore") if pathlib.Path("cloudflared.log").exists() else ""
    m = re.search(r"https://[-a-zA-Z0-9.]+\.trycloudflare\.com", text)
    if m:
        public_url = m.group(0)
        break

if not public_url:
    print(pathlib.Path("cloudflared.log").read_text(errors="ignore")[-4000:])
    raise RuntimeError("Không lấy được Cloudflare Tunnel URL.")

print("✅ Mở ComicVideoAI tại:", public_url)
print("Log app:", f"{PROJECT_DIR}/comicvideoai-dev.log")
print("Log tunnel:", f"{PROJECT_DIR}/cloudflared.log")


## Dừng server

Chạy cell dưới nếu muốn dừng Vite, OmniVoice hoặc Cloudflare Tunnel.


In [ ]:
#@title Dừng các tiến trình nền
import pathlib, subprocess, os
os.chdir(PROJECT_DIR)
for pid_file in ["comicvideoai-dev.pid", "omnivoice.pid", "cloudflared.pid"]:
    p = pathlib.Path(pid_file)
    if p.exists():
        pid = p.read_text().strip()
        if pid:
            subprocess.run(f"kill {pid} 2>/dev/null || true", shell=True)
            print("Stopped", pid_file, pid)
